##Placement assisant tool using LangChain

In [ ]:
!pip install -q langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.4 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd

from langchain_core.tools import tool
from langchain_groq import ChatGroq

In [ ]:
df = pd.read_csv("students.csv")

print(df)

   student_id     name branch  cgpa  backlogs  grad_year
0           1     Arun    CSE   8.7         0       2027
1           2    Priya    EEE   9.1         0       2027
2           3  Karthik    ECE   7.8         1       2027
3           4    Divya    CSE   8.3         0       2027
4           5    Rahul    EEE   7.2         0       2027
5           6    Sneha     IT   9.3         0       2027
6           7   Vikram    ECE   8.0         2       2027
7           8    Meena    CSE   9.0         0       2027
8           9     Ajay    EEE   8.5         0       2027
9          10    Nisha     IT   7.9         1       2027


In [ ]:
os.environ["GROQ_API_KEY"] = ""

In [ ]:
llm = ChatGroq(
    temperature=0,
    max_tokens=1000,
    timeout=10,
    groq_api_key=os.environ["GROQ_API_KEY"],
    model_name="openai/gpt-oss-20b"
)

print("LLM created successfully!")

LLM created successfully!


In [ ]:
from langchain_core.tools import tool

@tool
def get_student(student_id: int) -> str:
    """Get the details of a student using the student ID."""

    student = df[df["student_id"] == student_id]

    if student.empty:
        return f"No student found with ID {student_id}."

    return student.to_string(index=False)

In [ ]:
@tool
def find_eligible_students(min_cgpa: float) -> str:
    """Find students whose CGPA is greater than or equal to the given minimum CGPA."""

    eligible = df[df["cgpa"] >= min_cgpa]

    if eligible.empty:
        return f"No students found with CGPA >= {min_cgpa}."

    return eligible.to_string(index=False)

In [ ]:
tools = [
    get_student,
    find_eligible_students
]

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools
)

In [ ]:
question = input("Ask something about the students: ")

result = agent.invoke({
    "messages": [
        {"role": "user", "content": question}
    ]
})

# Get the final answer
answer = result["messages"][-1].content

if isinstance(answer, list):
    answer = "\n".join(
        item["text"]
        for item in answer
        if isinstance(item, dict) and item.get("type") == "text"
    )

print("\n" + "=" * 60)
print(question)
print("=" * 60)
print(answer)
print("=" * 60)

Ask something about the students: What is the information of student 2?

What is the information of student 2?
**Student 2 – Priya**

| Field      | Value |
|------------|-------|
| **Student ID** | 2 |
| **Name**        | Priya |
| **Branch**      | EEE (Electrical & Electronics Engineering) |
| **CGPA**        | 9.1 |
| **Backlogs**    | 0 |
| **Expected Graduation Year** | 2027 |
